# Main pretraining run (weeks 2–4) — 1B tokens
Run in several 12h sessions. **Before each new session**: add the previous session's output (Save Version → output) as an input dataset and set `PREV` below. Training resumes automatically.

Then run the same notebook with `PRESET='dense-6'` and `PRESET='dense-16'` for the baselines.

In [ ]:
# Settings: Accelerator = GPU T4 x2 (we use one), Internet = ON, Persistence = Files
REPO = "https://github.com/preranam05/loopthink"   # <- after you push this repo
!git clone -q $REPO /kaggle/working/loopthink || (cd /kaggle/working/loopthink && git pull -q)
%cd /kaggle/working/loopthink
!pip -q install tokenizers datasets huggingface_hub
import torch; print(torch.__version__, torch.cuda.get_device_name(0), "bf16:", torch.cuda.is_bf16_supported())

In [ ]:
PRESET = 'looped-27m'   # 'dense-6' (param-matched) | 'dense-16' (FLOP-matched)
DATA   = '/kaggle/input/fineweb-tok-1b'   # output of prepare_pretrain_data.py --tokens 1e9, saved as a Dataset
PREV   = ''   # e.g. '/kaggle/input/loopthink-pretrain-v1' from the last session
OUT    = f'/kaggle/working/pretrain_{PRESET}'

In [ ]:
# one-off, in a CPU notebook, then save as Dataset 'fineweb-tok-1b':
# !python scripts/prepare_pretrain_data.py --out /kaggle/working/tok1b --tokens 1e9

In [ ]:
!python -m loopthink.pretrain --data_dir $DATA --out $OUT --preset $PRESET --tokens 1e9 --precision fp16 --resume --resume_from "$PREV" --max_hours 11.3

In [ ]:
import json, matplotlib.pyplot as plt
L=[json.loads(l) for l in open(f'{OUT}/log.jsonl')]
tr=[r for r in L if 'loss' in r]; ev=[r for r in L if 'val_loss_r1' in r]
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].plot([r['step'] for r in tr],[r['loss'] for r in tr]); ax[0].set_title('train loss')
for k in [k for k in ev[-1] if k.startswith('val_loss')] if ev else []: ax[1].plot([r['step'] for r in ev],[r[k] for r in ev],label=k)
ax[1].legend(); ax[1].set_title('val loss by loops (test-time scaling)')
print('max activation seen:', max(r['max_act'] or 0 for r in tr))